# Pretrained Fake News Detector Audit

Este notebook aplica um modelo detector de fake news ja treinado sobre textos reescritos.

Fluxo:
1) Carrega datasets reescritos
2) Executa inferencia no texto reescrito
3) (Opcional) Compara com labels reais, se existirem
4) Salva resultados por dataset e consolidado

## 1) Imports

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## 2) Configuracao

In [ ]:
# Pasta com CSVs reescritos exportados no notebook de simulacao
run_id_env = os.getenv("RUN_ID")
if run_id_env:
    RUN_ID = run_id_env
    INPUT_DIR = Path("../output/runs") / RUN_ID / "rewritten"
    AUDIT_OUTPUT_DIR = Path("../output/runs") / RUN_ID / "audit" / "PreTrainedAudit"
else:
    RUN_ID = None
    INPUT_DIR = Path("../output/rewritten")
    AUDIT_OUTPUT_DIR = Path("../output/audit/PreTrainedAudit")

# Selecao global do dataset:
# - "ALL" para processar todos os CSVs da pasta INPUT_DIR
# - nome de arquivo (ex.: "science_rewritten_df.csv") para processar apenas um
DATASET_SELECTOR = "ALL"

# Coluna de texto usada na inferencia
REWRITTEN_COLUMN = "rewritten_news"

# Coluna de label real (opcional). Se None, nao calcula classificacao supervisionada
LABEL_COLUMN = None  # ex.: "label"

# Detector de fake news pre-treinado
DETECTOR_MODEL = "jy46604790/Fake-News-Bert-Detect"

MAX_ROWS_PER_DATASET = 1000

# Pasta de saida
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuracao carregada.")
if RUN_ID:
    print(f"RUN_ID: {RUN_ID}")
else:
    print("RUN_ID nao definido. Usando pastas padrao em ../output.")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"DATASET_SELECTOR: {DATASET_SELECTOR}")
print(f"REWRITTEN_COLUMN: {REWRITTEN_COLUMN}")
print(f"DETECTOR_MODEL: {DETECTOR_MODEL}")
print(f"AUDIT_OUTPUT_DIR: {AUDIT_OUTPUT_DIR}")

## 3) Funcoes utilitarias

In [ ]:
from utils.bert_audit_functions import (
    pretrained_fake_news_detector_prediction,
    read_dataset,
)

## 4) Carregar datasets

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Pasta de entrada nao encontrada: {INPUT_DIR}")

if DATASET_SELECTOR.strip().upper() == "ALL":
    selected_files = sorted(INPUT_DIR.glob("*.csv"), key=lambda p: p.name.lower())
else:
    selected_file = INPUT_DIR / DATASET_SELECTOR
    if not selected_file.exists():
        raise FileNotFoundError(f"Arquivo selecionado nao encontrado: {selected_file}")
    selected_files = [selected_file]

if not selected_files:
    raise FileNotFoundError(f"Nenhum CSV encontrado em: {INPUT_DIR}")

dataset_payloads = []
for file_path in selected_files:
    df = read_dataset(file_path).copy()

    if REWRITTEN_COLUMN not in df.columns:
        raise ValueError(f"Coluna de texto nao encontrada em {file_path.name}: {REWRITTEN_COLUMN}")

    df[REWRITTEN_COLUMN] = df[REWRITTEN_COLUMN].fillna("").astype(str).str.strip()
    df = df[df[REWRITTEN_COLUMN] != ""].copy()
    df = df.head(MAX_ROWS_PER_DATASET).copy()

    dataset_payloads.append({
        "file_path": file_path,
        "file_name": file_path.name,
        "dataset_name": file_path.stem,
        "df": df,
    })

print(f"Datasets selecionados: {len(dataset_payloads)}")
for payload in dataset_payloads:
    print(f"- {payload['file_name']}: {len(payload['df'])} linhas")

dataset_payloads[0]["df"][[REWRITTEN_COLUMN]].head()

## 5) Carregar detector e executar inferencia

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(DETECTOR_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(DETECTOR_MODEL)
model.eval()

all_predictions = []
summary_rows = []

for payload in dataset_payloads:
    current_df = payload["df"].copy()
    prediction_rows = []

    for idx, text in current_df[REWRITTEN_COLUMN].items():
        pred = pretrained_fake_news_detector_prediction(tokenizer, model, text)
        prediction_rows.append(
            {
                "row_index": int(idx),
                "prediction_id": pred["prediction_id"],
                "prediction_label": pred["prediction_label"],
                "prediction_confidence": pred["prediction_confidence"],
            }
        )

    pred_df = pd.DataFrame(prediction_rows)
    result_df = current_df.join(pred_df.set_index("row_index"), how="left")
    result_df["source_file"] = payload["file_name"]
    result_df["dataset_name"] = payload["dataset_name"]

    all_predictions.append(result_df)

    fake_rate = (result_df["prediction_label"].str.lower().eq("fake").mean() if len(result_df) > 0 else 0.0)
    summary_rows.append(
        {
            "dataset_name": payload["dataset_name"],
            "source_file": payload["file_name"],
            "rows": int(len(result_df)),
            "fake_rate": float(fake_rate),
        }
    )

predictions_all_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows).sort_values("fake_rate", ascending=False)

print("Resumo por dataset:")
display(summary_df)

predictions_all_df[["dataset_name", REWRITTEN_COLUMN, "prediction_label", "prediction_confidence"]].head(10)

## 6) (Opcional) Relatorio de classificacao

In [ ]:
if LABEL_COLUMN and LABEL_COLUMN in predictions_all_df.columns:
    y_true = predictions_all_df[LABEL_COLUMN].astype(str)
    y_pred = predictions_all_df["prediction_label"].astype(str)
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    print("LABEL_COLUMN nao definida/encontrada. Pulando classification_report.")

## 7) Salvar resultados

In [ ]:
if predictions_all_df.empty:
    raise ValueError("Nenhum resultado foi gerado.")

per_file_outputs = []
for dataset_name, dataset_df in predictions_all_df.groupby("dataset_name", dropna=False):
    output_file = AUDIT_OUTPUT_DIR / f"{dataset_name}_pretrained_fake_news_predictions.csv"
    dataset_df.to_csv(output_file, index=False)
    per_file_outputs.append(output_file)

combined_output = AUDIT_OUTPUT_DIR / "all_datasets_pretrained_fake_news_predictions.csv"
predictions_all_df.to_csv(combined_output, index=False)

summary_output = AUDIT_OUTPUT_DIR / "pretrained_fake_news_summary.csv"
summary_df.to_csv(summary_output, index=False)

print("Arquivos salvos:")
for output_file in per_file_outputs:
    print(f"- {output_file}")
print(f"- {combined_output}")
print(f"- {summary_output}")